# RAG 101 - Gemini + Pinecone + LangChain

> Build a complete Retrieval-Augmented Generation pipeline with one PDF, visible intermediate results, and very little code.

**What you’ll learn**  
1. What RAG is and why it’s useful  
2. How to index your docs in Pinecone using Gemini embeddings  
3. How to retrieve the most relevant chunks for a user’s question  
4. How to let Gemini read those chunks and answer — with sources

**Stack**  
- **LLM:** Google Gemini 
- **Vector DB:** Pinecone  
- **Framework:** LangChain  
- **Language:** Python

> You’ll only need two API keys: **Google Generative AI** and **Pinecone**. The language model generates the answer, while Pinecone stores and retrieves the document vectors.

```text
                ┌────────────────────┐
                │      Client        │
                │ (User Question)    │
                └────────┬───────────┘
                         │
                         ▼
                ┌──────────────────────┐
                │    Framework         │
                │ (Python + LangChain) │
                └────────┬─────────────┘
                         │
          (1) Semantic Search from Question Embedding
                         │
                         ▼
                ┌────────────────────┐
                │   Vector DB        │
                │   (Pinecone)       │
                │ Stores document    │
                │ embeddings         │
                └────────┬───────────┘
                         │
          (2) Retrieve Top Relevant Chunks
                         │
                         ▼
                ┌────────────────────┐
                │  Gemini (LLM)      │
                │  Reads context +   │
                │  question → Answer │
                └────────┬───────────┘
                         │
          (3) Return Final Response
                         │
                         ▼
                ┌────────────────────┐
                │     Client         │
                │ (Answer Displayed) │
                └────────────────────┘


## 0) Prerequisites

- Create accounts and get API keys:
  - **Google Generative AI**: https://ai.google.dev/
  - **Pinecone**: https://www.pinecone.io/
- Copy `.env.example` to `.env` and add your keys.
- Never commit the real `.env` file.
- Keep `ResumeBook.pdf` beside this notebook because the example uses a simple relative path.
- Make sure you’re on Python 3.10+.


## 1) Install libraries 

Run this once per environment.


In [ ]:
%pip install -q -r requirements.txt

## 2) Load API keys safely

- `load_dotenv(".env")` reads the local `.env` file.
- The keys are placed in this notebook session but never printed.
- If a key is missing, `getpass` asks for it using a hidden prompt.


In [ ]:
import os, getpass
from dotenv import load_dotenv

load_dotenv(".env")

os.environ['GOOGLE_API_KEY'] = os.getenv('GOOGLE_API_KEY') or getpass.getpass('Enter GOOGLE_API_KEY: ')
os.environ['PINECONE_API_KEY'] = os.getenv('PINECONE_API_KEY') or getpass.getpass('Enter PINECONE_API_KEY: ')

print("API keys loaded successfully")
print("  ✓ Google Gemini")
print("  ✓ Pinecone")

## 3) The RAG mental model

**Problem:** A language model can answer general questions, but it does not automatically know your private documents. When evidence is missing, it may guess.  
**Open-book-test idea:** first retrieve the most relevant passages, then let the model answer with those passages in front of it.  

A SQL database is designed for exact rows, IDs, and calculations. A vector database is designed for semantic similarity: it can match `resume preparation advice` with passages that say `writing your resume` even when the wording differs.


## 4) Load our knowledge base

Our knowledge base is `ResumeBook.pdf`, a 54-page guide covering the hiring process, resume structure, customization, cover letters, examples, and job-search preparation.

This cell extracts the PDF's text and wraps it as one source document. The next section will split that long source into smaller retrieval units.


In [ ]:
# Extract text from PDF
import logging
import pypdf

logging.getLogger('pypdf').setLevel(logging.ERROR)

def extract_pdf_text(pdf_path):
    """Extract text from PDF file"""
    text = ""
    with open(pdf_path, 'rb') as file:
        pdf_reader = pypdf.PdfReader(file)
        for page in pdf_reader.pages:
            text += page.extract_text() + "\n"
    return text

# Extract text from your ResumeBook.pdf
pdf_text = extract_pdf_text("ResumeBook.pdf")

# Create documents from PDF content
docs = [
    {
        "id": "resume-book",
        "text": pdf_text
    }
]

word_count = len(pdf_text.split())
preview = " ".join(pdf_text[:240].split())

print("PDF loaded successfully")
print("  File: ResumeBook.pdf")
print(f"  Documents: {len(docs)}")
print(f"  Characters: {len(pdf_text):,}")
print(f"  Words: {word_count:,}")
print(f"\nPreview: {preview}...")

## 5) Chunk the document

Sending all 54 pages for every question would be expensive and noisy. We split the guide into 400-word chunks with a 40-word overlap.

The overlap repeats a small amount of text between neighboring chunks so an important idea is less likely to be cut exactly at a boundary. Each chunk keeps a stable ID and its source name.


In [ ]:
from typing import List, Dict

def chunk_text(text: str, chunk_size: int = 400, overlap: int = 40) -> List[str]:
    """Split text into overlapping chunks for better retrieval"""
    words = text.split()
    chunks = []
    start = 0
    while start < len(words):
        end = min(len(words), start + chunk_size)
        chunks.append(' '.join(words[start:end]))
        if end == len(words): break
        start = end - overlap
    return chunks

# Process the PDF document into chunks
chunks: List[Dict] = []
for d in docs:
    # Create chunks from the PDF text
    text_chunks = chunk_text(d["text"])
    for i, ch in enumerate(text_chunks):
        chunks.append({
            "id": f'{d["id"]}-{i}',
            "text": ch,
            "source": d["id"]
        })

first_chunk_preview = " ".join(chunks[0]["text"][:240].split())

print("Chunking complete")
print(f"  Total chunks: {len(chunks)}")
print("  Chunk size: 400 words")
print("  Overlap: 40 words")
print(f"  First chunk ID: {chunks[0]['id']}")
print(f"\nFirst chunk preview: {first_chunk_preview}...")

## 6) Ingestion: embed and store

This is the indexing phase that normally runs when documents are added or updated:

1. Connect to Pinecone.
2. Create the serverless index if it does not already exist.
3. Ask Gemini to convert each chunk into a 768-number embedding.
4. Store the vector together with the original text and source metadata.

The chunk IDs are stable, so rerunning the cell updates the same records instead of creating duplicates.


In [ ]:
import os
from pinecone import Pinecone, ServerlessSpec
from langchain_google_genai import GoogleGenerativeAIEmbeddings

PINECONE_API_KEY = os.environ['PINECONE_API_KEY']
pc = Pinecone(api_key=PINECONE_API_KEY)

index_name = "rag-demo-gemini"
embedding_model = "gemini-embedding-2"
embedding_dimension = 768
# Create the index if it doesn't exist
existing_indexes = [i.name for i in pc.list_indexes()]
index_was_created = index_name not in existing_indexes
if index_was_created:
    pc.create_index(
        name=index_name,
        dimension=embedding_dimension,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )

index = pc.Index(index_name)

# Keep 768 dimensions so the vectors match our Pinecone index.
embeddings = GoogleGenerativeAIEmbeddings(
    model=embedding_model,
    output_dimensionality=embedding_dimension,
)

# Build vectors for upsert
vectors = []
print(f"Creating {len(chunks)} semantic embeddings...")
for position, c in enumerate(chunks, start=1):
    vec = embeddings.embed_query(c["text"])  # returns a 768-dim list
    vectors.append({
        "id": c["id"],
        "values": vec,
        "metadata": {"text": c["text"], "source": c["source"]}
    })
    if position % 5 == 0 or position == len(chunks):
        print(f"  Embedded {position} of {len(chunks)} chunks")

# Upsert to Pinecone
index.upsert(vectors=vectors)

index_status = "created now" if index_was_created else "already existed"
print("\nVector knowledge base ready")
print(f"  Pinecone index: {index_name} ({index_status})")
print(f"  Embedding model: {embedding_model}")
print(f"  Dimensions: {embedding_dimension}")
print("  Similarity metric: cosine")
print(f"  Vectors stored: {len(vectors)}")

## 7) Generation: retrieve first, answer second

When a user asks a question, we embed the question with the same embedding model and ask Pinecone for the three closest chunks (`Top-K = 3`).

Those chunks become the `context` inside the prompt. Gemini must answer only from that context, say `I don't know` when the evidence is missing, and include the source name. The retrieved context is printed so you can inspect what the model actually received.


In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from pinecone import Pinecone

# Init Pinecone
pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])
index_name = "rag-demo-gemini"
index = pc.Index(index_name)

# LangChain embeddings for querying
embeddings = GoogleGenerativeAIEmbeddings(
    model=embedding_model,
    output_dimensionality=embedding_dimension,
)

# LLM for answering
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",  # cheap + fast; swap to gemini-1.5-pro for higher quality
    temperature=0.2,
)

# Prompt template
prompt = PromptTemplate(
    input_variables=["question", "context"],
    template=(
        """You are a helpful assistant that answers using only the provided context.

Context:
{context}

Question: {question}

Rules:
- Be concise and clear.
- If the answer is not in the context, say you don't know.
- Cite sources at the end as [source:id]."""
    ),
)

def retrieve_docs(question, k=3):
    """Retrieve relevant documents from Pinecone using direct query"""
    # Generate embedding for the question
    query_embedding = embeddings.embed_query(question)
    
    # Query Pinecone directly
    results = index.query(
        vector=query_embedding,
        top_k=k,
        include_metadata=True
    )
    
    # Format results for the prompt
    docs = []
    for i, match in enumerate(results.matches, 1):
        source = match.metadata.get("source", "unknown")
        text = match.metadata.get("text", "")
        score = match.score or 0
        docs.append(
            f"Match {i} | Similarity: {score:.3f} | Chunk: {match.id} | Source: {source}\n"
            f"{text}"
        )
    
    return "\n\n".join(docs)

def ask_question(question):
    """Ask a question and get an answer using RAG"""
    # Retrieve relevant context
    context = retrieve_docs(question)
    print("\nRetrieved evidence")
    print("Pinecone selected the three passages most similar to the question.\n")
    print(context)
    # Format the prompt
    formatted_prompt = prompt.format(question=question, context=context)
    
    # Get answer from LLM
    response = llm.invoke(formatted_prompt)
    return response

print("RAG query engine ready")
print("  Query → Embed → Retrieve top 3 → Generate answer")

## 8) Run the complete RAG query

The first question asks for resume-preparation advice. Watch the output in two parts: first the three retrieved chunks, then Gemini's grounded answer.

After that, try questions such as:
- *Give me tips to prepare a resume*  
- *How many pages should my Resume have?*  
- *Do I need a cover letter?*


In [ ]:
question = "Give me tips to prepare a resume"

print("User question")
print(f"  {question}")

response = ask_question(question)

print("\nGrounded answer")
print(response.content)

## 9) What the notebook accomplished

1. **Chunk** - split a long PDF into roughly 20 manageable passages.  
2. **Embed** - convert each passage into a 768-dimensional representation of meaning.  
3. **Store** - save vectors, original text, and source metadata in Pinecone.  
4. **Retrieve** - embed the question and select the three closest chunks.  
5. **Generate** - give the question and retrieved evidence to Gemini.  
6. **Verify** - inspect the printed context and source label.

That is the core RAG loop: retrieve evidence first, then generate the answer. Production systems add stronger chunking, page-level citations, evaluation, access control, and monitoring.
